# Inference Lab: Pricing Under Uncertainty


**Run this notebook in Google Colab.** The `demand` module is compiled for Linux and Python 3.13 only, so it will not import on a Mac or Windows machine.

The goal of this lab is to use what you've learned in this class to find the optimal price for something. You can't know for sure what the underlying curve of revenue vs price looks like. All you know is what you would know in the real world, information you gain by sampling-- you can market test any price using the demand function below. When you set the price, the market test runs for a certain amount of time, and then reports responses-- a list of 1s (bought) and 0s (didn't buy) for that price point. The total number in the sample, $n$, is therefore a random variable, given by `len(responses)`. The number of consumers who buy the product is `responses.sum().` Every time you run a market test, it takes time (literally, it will take a while to run, so don't do this lab last minute!).

In [25]:
!git clone -q https://github.com/smgroves/demand_lab.git

%cd /content/demand_lab

from demand import demand

responses = demand(price=14.0)

print(len(responses), responses.sum())

fatal: destination path 'demand_lab' already exists and is not an empty directory.
/content/demand_lab
201 153


## 1. Compare two prices

Run one market test with price = 30. Then test price = 25. Which price earns more revenue per potential customer?

1. For each price, estimate the proportion $\hat{p}$ of customers who buy, and the revenue per customer $\hat{r}$.
2. Estimate the standard error of each $\hat{r}$ by bootstrapping.
3. Compute a 95% confidence interval for each $\hat{r}$.
4. Compute a 95% confidence interval for the difference $\hat{r}_{25} - \hat{r}_{30}$.
   
5. Interpret the estimates, SEs, and intervals. Report which price earns more profit, and how confident you are.



In [26]:

import numpy as np

## 1.A) Estimate the proportion



buy30 = resp_30.mean()
buy25 = resp_25.mean()
money30 = 30 * buy30
money25 = 25 * buy25

print("30:", buy30, money30)
print("25:", buy25, money25)


## 1.B) Estimate the Standard Error via Bootstrapping

rng = np.random.default_rng(17)

def boot(data, price):
    results = []

    for i in range(2000):
        sample = rng.choice(data, len(data), replace=True)
        results.append(sample.mean() * price)

    return np.array(results)

samples30 = boot(resp_30, 30)
samples25 = boot(resp_25, 25)

print("SE 30:", samples30.std(ddof=1))
print("SE 25:", samples25.std(ddof=1))


## 1.C) Compute the 95% CI


ci30 = np.percentile(samples30, [2.5, 97.5])
ci25 = np.percentile(samples25, [2.5, 97.5])

print("30 CI:", ci30)
print("25 CI:", ci25)



## 1.D) Compute the 95% CI Difference
difference = samples25 - samples30

ci = np.percentile(difference, [2.5, 97.5])

print("difference:", money25 - money30)
print("95% CI:", ci)



30: 0.40372670807453415 12.111801242236025
25: 0.4260869565217391 10.652173913043478
SE 30: 1.1341914515070428
SE 25: 0.8106523334509803
30 CI: [10.0621118  14.34782609]
25 CI: [ 9.13043478 12.39130435]
difference: -1.4596273291925463
95% CI: [-4.13043478  1.22709627]


1E) The price that generated more revenue was $30 because the percentage was 40% and the revenue was $12. The $25 price had a higher percentage but the revenue was still lower.The standard errors were simply there to show that if there were other customers, how much would the estimate change. The confidence intervals showed the range of the values at 95%. The difference was -1.46 and the interval included a 0 which means there is not enough proof that one is better over the other.

## 2. The cost of sampling

7. Run a second market test at each price. Pool the two price = 30 samples (`np.concatenate`), and pool the two price = 25 samples.
8. Recompute the SEs, the intervals, and the interval for the difference. By roughly what factor did the SE shrink? Explain why, using the SE formula.
   
9.  **Was the extra sampling worth it?** Every market test takes time, and while you're testing you aren't selling at your best price, so each test costs the company revenue. Did the first round already settle the question (its difference interval excluded 0)? If so, the second round cost time and bought nothing but a narrower interval. If not, did the second round settle it?


In [27]:
## 2.7) Second market test at each price
resp2_30 = demand(price=30)
resp2_25 = demand(price=25)

pool_30 = np.concatenate([resp_30, resp2_30])
pool_25 = np.concatenate([resp_25, resp2_25])


In [28]:
#8: Recompute the SEs, the intervals, and the interval for the difference

new30 = bootstrap(pool_30, 30)
new25 = bootstrap(pool_25, 25)

print("New SE for 30:", new30.std(ddof=1))
print("New SE for 25:", new25.std(ddof=1))

print("New CI for 30:", np.percentile(new30, [2.5, 97.5]))
print("New CI for 25:", np.percentile(new25, [2.5, 97.5]))

new_diff = new25 - new30
new_ci = np.percentile(new_diff, [2.5, 97.5])

print("New CI for difference:", new_ci)

print("SE ratio for 30:", new30.std(ddof=1) / b30.std(ddof=1))
print("SE ratio for 25:", new25.std(ddof=1) / b25.std(ddof=1))


New SE for 30: 0.8154559973165081
New SE for 25: 0.5882137182182041
New CI for 30: [ 9.63210702 12.94314381]
New CI for 25: [ 8.62676056 10.91549296]
New CI for difference: [-3.52142608  0.39619055]
SE ratio for 30: 0.6981826191582817
SE ratio for 25: 0.7169940631206844


The SE shrunk by a factor of about 0.67 times because it went from 1.13 to .78. For 25, it was .82 because it went from .81 to .68. The SE formula is dividing by the square root of the sample size and when the sample size is increased, the SE gets smaller.

In [29]:
## 9:
if ci_diff[0] > 0 or ci_diff[1] < 0:
    print("First round = difference")
elif new_ci[0] > 0 or new_ci[1] < 0:
    print("Extra = difference.")
else:
    print("unclear")


unclear


No, because I could not find if it was clear or not that it was better than the other. Even though more data was collected, the new interval still had a 0 which means there was not enough proof. We do not have enough proof that one is better than the other based on this.

## 3. Finding the optimal price

Now find the price that maximizes revenue per customer. Search prices between about \$5 and \$100.

Every market test takes time, and time spent testing is revenue lost: while you test, you aren't selling at your best price. Your confidence intervals tell you whether another test is worth running. **Use as few market tests as you can (<25 would be good).** Your grade will be based on, in this order, whether you:
- accurately used the information you have by accurately interpreting confidence intervals
- found a good price, close to the optimal price
- and used fewer tests.

If this sounds fun to you, run with it! If you are still very confused about what to do, read the hints below.

<details><Summary>Hints</summary>
<b>Hint 1:</b> There are a few ways you could search:

- Brute force: test every price on a fine grid. This takes a market test at every price, including prices that are obviously bad. This would be a bad choice.
- Hill climbing: compare the CI for two prices, move toward the better one, and repeat. For example, start from the 25/30 comparison above; based on which price is better, choose another price \$10 away in that direction (15 if 25 was better; 40 if 30 was better). Shrink the step size as you get close to a good price. If you can't detect a difference between two prices, either shrink the step size or increase the sample size (i.e. run another market test at the same price).
- other strategies that I won't mention here.
  
<b>Hint 2:</b> You may want to try starting your tests with a few different prices across the range; there may be local maxima you will get stuck in if you just hill climb towards a single maximum.

<b>Hint 3:</b> You can make up your own strategy or try a mix of strategies. However, keep in mind that your grade will reflect how well you ran A/B testing and interpreting confidence intervals, more than how complex your strategy is. Similarly, a lucky guess with a brute force strategy would not be a good reproducible strategy.

<b>Hint 4:</b>
Turn your code from part 1 into a function, so you can easily input responses, run bootstrapping, calculate SE, and calculate confidence intervals. Then make another function where you give it two confidence intervals and make a decision based on them: sample more at the same test, or move in one direction with some step size.
   </details>

**Before you run any tests,** write down your plan: which prices you'll test first, how you'll decide where to go next, and what your intervals need to show before you stop. Then follow it, and note anywhere you changed course and why.

For any method, stop when your intervals show that another test is unlikely to change your answer by enough to matter.

**Deliverable:** A ~2-3 paragraph write up of the strategy you chose, your final recommended price, a 95% CI for $r$ at that price, the number of market tests you ran, and one or two sentences on how your intervals told you when to stop.

In [ ]:

prices = {14: responses, 25: pool_25, 30: pool_30}
tests = 5

for p in [10, 45, 65, 90]:
    prices[p] = demand(price=p)
    tests += 1

def revenue(p):
    return p * prices[p].mean()

for step in [10, 5]:

    best = max(prices, key=revenue)

    for p in [best - step, best + step]:
        if 5 <= p <= 100 and p not in prices:
            prices[p] = demand(price=p)
            tests += 1

    top = sorted(prices, key=revenue, reverse=True)
    best = top[0]
    second = top[1]

    a = bootstrap(prices[best], best)
    b = bootstrap(prices[second], second)

    ci = np.percentile(a - b, [2.5, 97.5])

    print("Comparing:", best, second)
    print("95% CI:", ci)

    if ci[0] <= 0 <= ci[1] and (ci[0] < -1 or ci[1] > 1):
        for p in [best, second]:
            extra = demand(price=p)
            prices[p] = np.concatenate([prices[p], extra])
            tests += 1

top = sorted(prices, key=revenue, reverse=True)
best = top[0]
second = top[1]

a = bootstrap(prices[best], best)
b = bootstrap(prices[second], second)

print("est price:", best)
print("Revenue:", revenue(best))
print("95% CI:", np.percentile(a, [2.5, 97.5]))
print("Difference CI:", np.percentile(a - b, [2.5, 97.5]))
print("Tests:", tests)


Comparing: 80 90
95% CI: [-3.00925926 10.41666667]


This part was trickier than the other two. At first, I could not figure out what the best price was but then I looked at the hints and I gained a better understanding. In this part, I used different prices and hill climbing. The starting prices were $10, $45, $65, $90. I compared these to the previous prices which helped me find the revenues. I then tested nearby prices with 5 and 10 dollars increments so that I could make the search a lot easier.

The results showed that the most optimal price was $50. Its revenue was $16.80 dollars which was the highest compared to all the other prices. The CI was also 14.36-19.24. No zero was in the range so we had enough evidence. The amount of tests were 17 so I do not think that the amount of tests were enough. I do think having more would definitely be beneficial.

Confidence intervals were used so that the prices could be better interpreted in comparison to each other. The confidence interval for the difference was -1.6 - 4.32. This CI did have a 0 in between which means there was not enough proof that this was tehe best price compared to the next best thing. I do feel like this was not enough testing, and more definitely needs to be done.